In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import matplotlib.pyplot as plt

PROJECTION_MONTHS = 12

In [ ]:
df = pd.read_csv("ml.csv")

df["service_date"] = pd.to_datetime(
    df["service_date"]
)

df = (
    df
    .sort_values(["vehicle_id", "service_date"])
    .reset_index(drop=True)
)

print(df.shape)
df.head()

In [ ]:
def parse_parts(value):

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        part.strip()
        for part in value.split(",")
        if part.strip()
    ]


df["parts_list"] = (
    df["parts_replaced"]
    .apply(parse_parts)
)
df.head()

In [ ]:
parts_catalog = {

    # -------------------------
    # Engine
    # -------------------------
    "Radiator": {
        "cost": 75,
        "labor_hours": 2.0
    },

    "Coolant Pump": {
        "cost": 45,
        "labor_hours": 2.0
    },

    "Thermostat": {
        "cost": 18,
        "labor_hours": 1.0
    },

    "Oil Seal": {
        "cost": 8,
        "labor_hours": 2.0
    },

    "Gasket": {
        "cost": 12,
        "labor_hours": 2.0
    },

    "Spark Plug": {
        "cost": 8,
        "labor_hours": 0.5
    },

    "Fuel Injector": {
        "cost": 35,
        "labor_hours": 1.5
    },

    # -------------------------
    # Brakes
    # -------------------------
    "Brake Pads": {
        "cost": 32,
        "labor_hours": 1.0
    },

    "Brake Disc": {
        "cost": 30,
        "labor_hours": 1.5
    },

    "Brake Hose": {
        "cost": 12,
        "labor_hours": 1.0
    },

    "Brake Fluid": {
        "cost": 4,
        "labor_hours": 0.5
    },

    # -------------------------
    # Electrical
    # -------------------------
    "Battery": {
        "cost": 45,
        "labor_hours": 0.5
    },

    "Alternator": {
        "cost": 85,
        "labor_hours": 2.0
    },

    "Starter Motor": {
        "cost": 70,
        "labor_hours": 2.0
    },

    # -------------------------
    # Transmission
    # -------------------------
    "Clutch Kit": {
        "cost": 140,
        "labor_hours": 4.0
    },

    "Gearbox": {
        "cost": 600,
        "labor_hours": 6.0
    },

    "Transmission Seal": {
        "cost": 15,
        "labor_hours": 2.0
    },

    "Transmission Fluid": {
        "cost": 25,
        "labor_hours": 1.0
    },

    # -------------------------
    # Suspension / Steering
    # -------------------------
    "Shock Absorber": {
        "cost": 55,
        "labor_hours": 2.0
    },

    "Control Arm": {
        "cost": 60,
        "labor_hours": 2.0
    },

    "Alignment Service": {
        "cost": 8,
        "labor_hours": 1.0
    },

    "Power Steering Pump": {
        "cost": 100,
        "labor_hours": 3.0
    },

    # -------------------------
    # Tires / Wheels
    # -------------------------
    "Tire": {
        "cost": 45,
        "labor_hours": 0.5
    },

    "Wheel Bearing": {
        "cost": 35,
        "labor_hours": 2.0
    },

    # -------------------------
    # Cooling
    # -------------------------
    "AC Compressor": {
        "cost": 180,
        "labor_hours": 3.0
    },

    "Refrigerant": {
        "cost": 15,
        "labor_hours": 1.0
    }
}

parts = list(parts_catalog.keys())

print("Number of parts:", len(parts))

In [ ]:
part_records = []

for _, row in df.iterrows():

    month = row["service_date"].to_period("M").to_timestamp()

    for part in row["parts_list"]:

        if part in parts_catalog:

            part_records.append({
                "month": month,
                "part": part,
                "quantity": 1
            })

parts_usage = pd.DataFrame(part_records)

print(parts_usage.head())

In [ ]:
monthly_demand = (
    parts_usage
    .groupby(["month", "part"])["quantity"]
    .sum()
    .reset_index()
)
monthly_demand.head(20)

In [ ]:
all_months = pd.date_range(
    start=monthly_demand["month"].min(),
    end=monthly_demand["month"].max(),
    freq="MS"
)

full_index = pd.MultiIndex.from_product(
    [all_months, parts],
    names=["month", "part"]
)

monthly_demand = (
    monthly_demand
    .set_index(["month", "part"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

monthly_demand["quantity"] = (
    monthly_demand["quantity"]
    .astype(int)
)

In [ ]:
demand_matrix = (
    monthly_demand
    .pivot(
        index="month",
        columns="part",
        values="quantity"
    )
    .fillna(0)
)

demand_matrix.head()

In [ ]:
part = "Battery"

plt.figure(figsize=(12, 5))

plt.plot(
    demand_matrix.index,
    demand_matrix[part]
)

plt.title(
    f"Historical Monthly Demand - {part}"
)

plt.xlabel("Month")
plt.ylabel("Units")

plt.tight_layout()
plt.show()

In [ ]:
def create_forecasting_features(
    demand_series
):

    data = pd.DataFrame({
        "month": demand_series.index,
        "demand": demand_series.values
    })

    # Calendar features
    data["month_number"] = (
        data["month"].dt.month
    )

    data["year"] = (
        data["month"].dt.year
    )

    # Cyclic representation of month
    data["month_sin"] = np.sin(
        2 * np.pi *
        data["month_number"] / 12
    )

    data["month_cos"] = np.cos(
        2 * np.pi *
        data["month_number"] / 12
    )

    # Lag features
    data["lag_1"] = data["demand"].shift(1)
    data["lag_2"] = data["demand"].shift(2)
    data["lag_3"] = data["demand"].shift(3)
    data["lag_6"] = data["demand"].shift(6)
    data["lag_12"] = data["demand"].shift(12)

    # Rolling demand
    data["rolling_3"] = (
        data["demand"]
        .shift(1)
        .rolling(3)
        .mean()
    )

    data["rolling_6"] = (
        data["demand"]
        .shift(1)
        .rolling(6)
        .mean()
    )

    data["rolling_12"] = (
        data["demand"]
        .shift(1)
        .rolling(12)
        .mean()
    )

    # Rolling volatility
    data["rolling_std_6"] = (
        data["demand"]
        .shift(1)
        .rolling(6)
        .std()
    )

    return data

In [ ]:
forecasting_data = {}

for part in parts:

    series = demand_matrix[part]

    forecasting_data[part] = (
        create_forecasting_features(series)
    )
forecasting_data["Battery"].head(15)

In [ ]:
feature_columns = [
    "month_number",
    "year",
    "month_sin",
    "month_cos",

    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",

    "rolling_3",
    "rolling_6",
    "rolling_12",
    "rolling_std_6"
]

In [ ]:
for part in parts:

    forecasting_data[part] = (
        forecasting_data[part]
        .dropna()
        .reset_index(drop=True)
    )

In [ ]:
TEST_MONTHS = 12
models = {}
predictions = {}
evaluation = {}

In [ ]:
for part in parts:

    data = forecasting_data[part]

    if len(data) <= TEST_MONTHS:
        print(
            f"Skipping {part}: "
            f"not enough data"
        )
        continue

    train = data.iloc[:-TEST_MONTHS]
    test = data.iloc[-TEST_MONTHS:]

    X_train = train[feature_columns]
    y_train = train["demand"]

    X_test = test[feature_columns]
    y_test = test["demand"]

    model = RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(X_test)

    # Demand cannot be negative
    y_pred = np.maximum(
        y_pred,
        0
    )

    models[part] = model

    predictions[part] = {
        "actual": y_test.values,
        "predicted": y_pred
    }

    evaluation[part] = {
        "MAE": mean_absolute_error(
            y_test,
            y_pred
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_test,
                y_pred
            )
        )
    }

In [ ]:
evaluation_df = (
    pd.DataFrame(evaluation)
    .T
    .sort_values("MAE")
)

evaluation_df

In [ ]:
part = "Battery"

actual = predictions[part]["actual"]
predicted = predictions[part]["predicted"]

plt.figure(figsize=(12, 5))

plt.plot(
    actual,
    label="Actual"
)

plt.plot(
    predicted,
    label="Predicted"
)

plt.title(
    f"{part} - Actual vs Predicted Demand"
)

plt.xlabel("Test Month")
plt.ylabel("Units")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def forecast_future(
    model,
    historical_series,
    months_ahead
):

    history = historical_series.copy()

    forecasts = []

    for _ in range(months_ahead):

        next_month = (
            history.index[-1]
            + pd.offsets.MonthBegin(1)
        )

        temp = pd.DataFrame(
            index=history.index
        )

        temp["demand"] = history.values

        temp["month_number"] = (
            temp.index.month
        )

        temp["year"] = (
            temp.index.year
        )

        temp["month_sin"] = np.sin(
            2 * np.pi *
            temp["month_number"] / 12
        )

        temp["month_cos"] = np.cos(
            2 * np.pi *
            temp["month_number"] / 12
        )

        temp["lag_1"] = (
            temp["demand"].shift(1)
        )

        temp["lag_2"] = (
            temp["demand"].shift(2)
        )

        temp["lag_3"] = (
            temp["demand"].shift(3)
        )

        temp["lag_6"] = (
            temp["demand"].shift(6)
        )

        temp["lag_12"] = (
            temp["demand"].shift(12)
        )

        temp["rolling_3"] = (
            temp["demand"]
            .shift(1)
            .rolling(3)
            .mean()
        )

        temp["rolling_6"] = (
            temp["demand"]
            .shift(1)
            .rolling(6)
            .mean()
        )

        temp["rolling_12"] = (
            temp["demand"]
            .shift(1)
            .rolling(12)
            .mean()
        )

        temp["rolling_std_6"] = (
            temp["demand"]
            .shift(1)
            .rolling(6)
            .std()
        )

        latest = temp.iloc[-1]

        X_future = pd.DataFrame(
            [{
                "month_number": next_month.month,
                "year": next_month.year,
                "month_sin": np.sin(
                    2 * np.pi *
                    next_month.month / 12
                ),
                "month_cos": np.cos(
                    2 * np.pi *
                    next_month.month / 12
                ),

                "lag_1": history.iloc[-1],
                "lag_2": history.iloc[-2],
                "lag_3": history.iloc[-3],
                "lag_6": history.iloc[-6],
                "lag_12": history.iloc[-12],

                "rolling_3": history.iloc[-3:].mean(),
                "rolling_6": history.iloc[-6:].mean(),
                "rolling_12": history.iloc[-12:].mean(),

                "rolling_std_6": history.iloc[-6:].std()
            }]
        )

        prediction = model.predict(
            X_future
        )[0]

        prediction = max(
            0,
            prediction
        )

        history.loc[next_month] = prediction

        forecasts.append({
            "month": next_month,
            "predicted_demand": prediction
        })

    return pd.DataFrame(forecasts)

In [ ]:
future_forecasts = {}

for part, model in models.items():

    series = demand_matrix[part]

    future_forecasts[part] = (
        forecast_future(
            model,
            series,
            PROJECTION_MONTHS
        )
    )

In [ ]:
projection_records = []

for part, forecast in future_forecasts.items():

    for _, row in forecast.iterrows():

        projection_records.append({
            "month": row["month"],
            "part": part,
            "predicted_demand": row[
                "predicted_demand"
            ]
        })

future_projection = pd.DataFrame(
    projection_records
)

In [ ]:
annual_projection = (
    future_projection
    .groupby("part")["predicted_demand"]
    .sum()
    .sort_values(
        ascending=False
    )
    .reset_index()
)

annual_projection

In [ ]:
annual_projection[
    "unit_cost_omr"
] = annual_projection["part"].map(
    lambda x: parts_catalog[x]["cost"]
)

annual_projection[
    "projected_cost_omr"
] = (
    annual_projection[
        "predicted_demand"
    ]
    *
    annual_projection[
        "unit_cost_omr"
    ]
)

annual_projection[
    "projected_cost_omr"
] = (
    annual_projection[
        "projected_cost_omr"
    ].round(2)
)

In [ ]:
SAFETY_FACTOR = 1.20
annual_projection[
    "recommended_stock"
] = np.ceil(
    annual_projection[
        "predicted_demand"
    ] * SAFETY_FACTOR
).astype(int)

In [ ]:
annual_projection = annual_projection[
    [
        "part",
        "predicted_demand",
        "recommended_stock",
        "unit_cost_omr",
        "projected_cost_omr"
    ]
]

annual_projection

In [ ]:
monthly_projection = (
    future_projection
    .pivot(
        index="month",
        columns="part",
        values="predicted_demand"
    )
    .round(2)
)

monthly_projection

In [ ]:
part = "Brake Pads"

historical = demand_matrix[
    part
]

future = future_forecasts[
    part
]

plt.figure(figsize=(13, 6))

plt.plot(
    historical.index,
    historical.values,
    label="Historical"
)

plt.plot(
    future["month"],
    future["predicted_demand"],
    label="Forecast"
)

plt.axvline(
    historical.index[-1],
    linestyle="--"
)

plt.title(
    f"{part} Demand Forecast"
)

plt.xlabel("Month")
plt.ylabel("Units")

plt.legend()

plt.tight_layout()
plt.show()